<a href="https://colab.research.google.com/github/safcleo12/CL-homework-rep/blob/main/exercises/HomeExercise2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Home Exericse 2: Word Embeddings and PyTorch
In this second home exercise, you will use the knowledge from Tutorial 4 to perform a more systematic evaluation of embeddings based on a small analogy dataset.

In this notebook, please complete all instructions starting with 👋 ⚒ in the code cell after the sign or provide your analysis in the text cell after the sign.

---

## **Part 1: Implementing a tiny Skip-gram in PyTorch**

The objective of Skipgram is to assign a high score to true neighbors of a center word and low scores to random words called negative samples. The paremeters of the embedding models are two embedding matrices, one that connects the input layer to the hidden layer (representing center words) and one that connects the hidden layer to the output layer (representing context words). Training is performed the exact same way as in Tutorial 4.

We need to install gensim so that we can load a corpus for training. Run the following code cell and then restart your runtime environment by clicking on Runtime in the menu and selecting Restart Runtime. This is necessary because of the downversioning of smart_open for gensim.

In [1]:
!pip install gensim
#Due to incompatibility between gensim smart_open and Colab pre-installed smart_open
!pip install "smart_open<7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 97.6 MB/s  0:00:00
  Attempting uninstall: smart_open
    Found existing installation: smart_open 9.0.2
    Uninstalling smart_open-9.0.2:
      Successfully uninstalled smart_open-9.0.2


We will use a cleaned English Wikipedia corpus of approximately 30 MB for this task. We will first get 1,000,000 tokens from the corpus and then calculate the 5,000 most frequent types that represents our vocabulary.

We need to create an index that maps each unique word (type) in the vocabulary to a unique number called `word2id` in the following code cell. Then, we assign each word in the corpus the corresponding id so that we can input one sentence after another into the model for training.

In [2]:
import itertools
import gensim.downloader as api
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import Counter


corpus = api.load("text8")
tokens = list(itertools.islice(itertools.chain.from_iterable(corpus), 1_000_000))
print(len(tokens), "tokens")

V = 5000
vocab = [w for w, _ in Counter(tokens).most_common(V)]
word2id = {w: i for i, w in enumerate(vocab)}
ids = torch.tensor([word2id.get(w, -1) for w in tokens]) # -1 represents unknown words, that is, words not in the vocabulary

print(f"Creating an index/lookup dictionary that maps words to ids: {word2id}")
print(f"Mapping all 1000000 tokens in the text to ids: {len(ids)}")

[==================================================] 100.0% 31.6/31.6MB downloaded
1000000 tokens
Creating an index/lookup dictionary that maps words to ids: {'the': 0, 'of': 1, 'and': 2, 'one': 3, 'in': 4, 'a': 5, 'to': 6, 'zero': 7, 'nine': 8, 'is': 9, 'two': 10, 'as': 11, 'eight': 12, 'three': 13, 'was': 14, 'by': 15, 'five': 16, 's': 17, 'that': 18, 'for': 19, 'four': 20, 'six': 21, 'seven': 22, 'with': 23, 'on': 24, 'his': 25, 'are': 26, 'from': 27, 'or': 28, 'it': 29, 'an': 30, 'he': 31, 'be': 32, 'which': 33, 'at': 34, 'this': 35, 'not': 36, 'also': 37, 'have': 38, 'but': 39, 'has': 40, 'were': 41, 'other': 42, 'had': 43, 'their': 44, 'its': 45, 'first': 46, 'american': 47, 'they': 48, 'some': 49, 'all': 50, 'been': 51, 'most': 52, 'who': 53, 'more': 54, 'many': 55, 'such': 56, 'd': 57, 'b': 58, 'after': 59, 'there': 60, 'into': 61, 'used': 62, 'can': 63, 'these': 64, 'world': 65, 'when': 66, 'new': 67, 'time': 68, 'than': 69, 'see': 70, 'i': 71, 'may': 72, 'only': 73, 'war': 74

Next we need to create our true neighbor pairs and our negative samples for training.

In [6]:
# (center, context) pairs are created by looking at d positions to the right and adding mirror pairs
window_size = 4
centers, contexts = [], []
for d in range(1, window_size + 1):
    a, b = ids[:-d], ids[d:]
    ok = (a >= 0) & (b >= 0)
    centers += [a[ok], b[ok]]
    contexts += [b[ok], a[ok]]
centers, contexts = torch.cat(centers), torch.cat(contexts)
print(len(centers), "training pairs")

# Negative distances are calculated as the probability of a specific word to be a negative sample with sample words by frequency^0.75 (as in word2vec)
counts = torch.bincount(ids[ids >= 0], minlength=V).float()
neg_dist = counts ** 0.75
neg_dist /= neg_dist.sum()
print(len(neg_dist), "each word gets a probability of picking that word as a negative, the more frequent the higher the probability")

5902828 training pairs
5000 each word gets a probability of picking that word as a negative, the more frequent the higher the probability


As a next step we need to compose the model with all its layers and the training step before we can actually train. We will keep it very small with only a dimensionality of 50. We have two embedding layers, one connecting the input layer to the hidden layer (center.weight) and the other connecting the hidden layer to the output layer (context.weight). Then we have one forward pass, where we need to obtain the vector for the center word `vc`, the vector for the context word `uo`, and the vector for the negative sample `un`. For every true pair, we draw K numbers of negative samples.

👋 ⚒ Exercise 1: Complete the forward pass of the following SkipGram model by adding the correct calculation of the true pair, which is the multiplication of two vectors that represent the true pair and then the sum over dim=1.

In [3]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, dim=50):
        super().__init__()
        self.center  = nn.Embedding(vocab_size, dim)
        self.context = nn.Embedding(vocab_size, dim)
        nn.init.uniform_(self.center.weight,  -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.context.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, c, o, neg):
        # c: (B,) centre ids   o: (B,) true context ids   neg: (B, K) negative ids
        vc = self.center(c)           # (B, dim)
        uo = self.context(o)          # (B, dim)
        un = self.context(neg)        # (B, K, dim)


        pos_score = (vc * uo).sum(dim=1)               # Your code here: complete the pos_score calculation
        neg_score = torch.bmm(un, vc.unsqueeze(2)).squeeze(2)   # (B, K), given

        return -(F.logsigmoid(pos_score) + F.logsigmoid(-neg_score).sum(dim=1)).mean()

After having defined the model, all the parameters (weights) in the model are random, which means it cannot identify words that are supposed to be neighbors. Run the following code cell to have a look at the performance before training.

In [4]:
def neighbors(word, emb_layer, k=8):
  emb_layer = emb_layer - emb_layer.mean(dim=0)                 # remove the common direction
  embeddings = emb_layer / emb_layer.norm(dim=1, keepdim=True)  # unit-length rows
  sims = embeddings @ embeddings[word2id[word]]                 # calculate cosine with every word
  top = sims.topk(k + 1).indices[1:].tolist()                   # keep only the top k words and return them
  return [vocab[t] for t in top]

model = SkipGram(V)
# The loaded corpus was already in lowercase
for w_ in ["king", "france", "music"]:
  print(w_, neighbors(w_, model.center.weight.detach()))

king ['g', 'vegetables', 'beat', 'tone', 'coast', 'action', 'learning', 'answers']
france ['played', 'manner', 'electoral', 'sources', 'driven', 'star', 'arguably', 'genetic']
music ['underlying', 'wife', 'pain', 'highway', 'baseball', 'absolute', 'stored', 'axis']


Then we need to perform the actual training step.

👋 ⚒ Exercise 2: Complete the loss and optimization steps in the following training loop.

In [7]:
model = SkipGram(V)
opt = torch.optim.Adam(model.parameters(), lr=0.003)
# K is the number of negative samples we draw from the vocabulary (bag of words)
# Batch size is the number of (center, context) word pairs processed in one training and weight update step
K, batch_size = 5, 8192
history = []

for epoch in range(3):
    perm = torch.randperm(len(centers))
    total, n = 0.0, 0
    losses = []
    for s in range(0, len(perm), batch_size):
        idx = perm[s:s + batch_size]
        c, o = centers[idx], contexts[idx]
        neg = torch.multinomial(neg_dist, len(idx) * K, replacement=True).view(len(idx), K)

        # Your code here for loss and optimization
        # To calculate the loss you need to perform a forward pass by using model() and
        # handing over the right variables for a forward pass

        loss = model(c, o, neg)
        opt.zero_grad()
        loss.backward()
        opt.step()


        losses.append(loss.item())

        total += loss.item(); n += 1

    history.append(total / n)
    print(f"Epoch {epoch}: Loss {history[-1]:.4f}")
    print(f"To take a look at all recorded loss per epoch, you can print losses: {losses}")

Epoch 0: Loss 2.5639
To take a look at all recorded loss per epoch, you can print losses: [4.158885478973389, 4.158812522888184, 4.1585798263549805, 4.158007621765137, 4.157043933868408, 4.155419826507568, 4.152924537658691, 4.149725914001465, 4.145275592803955, 4.13982629776001, 4.132797718048096, 4.1250505447387695, 4.115140914916992, 4.104443073272705, 4.091692924499512, 4.077065467834473, 4.061258792877197, 4.043797969818115, 4.024289131164551, 4.00351619720459, 3.9810657501220703, 3.9543867111206055, 3.9282710552215576, 3.899229049682617, 3.869995594024658, 3.8392467498779297, 3.804938316345215, 3.7689125537872314, 3.735893726348877, 3.700953483581543, 3.6579794883728027, 3.620150089263916, 3.5830328464508057, 3.539522886276245, 3.498673677444458, 3.4572370052337646, 3.420027256011963, 3.3749260902404785, 3.331259250640869, 3.292351722717285, 3.25337553024292, 3.210564613342285, 3.17647647857666, 3.13455867767334, 3.1031837463378906, 3.0650458335876465, 3.0294930934906006, 2.99799

---

## **Part 2: Evaluating your Skipgram model**

👋 ⚒ Exercise 3: Use the neighbors function from the previous part to test which words are the nearest neighbors of the following words. Do the provided nearest neighbors have a closer relation to the word than before training?



In [8]:
words = ["king", "france", "music", "water", "computer"]

for w in words:
  print(w, neighbors(w, model.center.weight.detach()))

king ['sultan', 'iv', 'bishop', 'prince', 'duke', 'iii', 'vi', 'philip']
france ['spain', 'portugal', 'bavaria', 'invaded', 'russia', 'belgium', 'italy', 'hungary']
music ['pop', 'anime', 'musical', 'cinema', 'producer', 'television', 'dance', 'fantasy']
water ['temperatures', 'fresh', 'snow', 'methane', 'unstable', 'salt', 'chain', 'ranges']
computer ['video', 'computers', 'portable', 'software', 'macintosh', 'intel', 'adventure', 'microsoft']


Question: Do the provided nearest neighbors have a closer relation to the word than before training?
Answer:
In short: yes. The nearest neighbors are closely related to the words after training. For example "king" is associated with "prince" and "duke" now while "france" is associated with countries like "spain" and "portugal". Most neighbors were unrelated and random before the training. This shows that the model learned word relationships during the training.

👋 ⚒ Exercise 4: Add another code cell here were you continue training the same model for another 3 epochs. You need to copy most of the code from the end of Part 1 but not all of it.

Mind that the loss of Epoch 0 should be lower than the previous loss of Epoch 2 in Part 1, since you continue the training from that last epoch.

In [10]:
for epoch in range(3):
  perm = torch.randperm(len(centers))
  total, n = 0.0, 0
  losses = []

  for s in range(0, len(perm), batch_size):
    idx = perm[s:s + batch_size]
    c, o = centers[idx], contexts[idx]
    neg = torch.multinomial(neg_dist, len(idx) * K, replacement=True).view(len(idx), K)

    loss = model(c, o, neg)
    opt.zero_grad()
    loss.backward()
    opt.step()

    losses.append(loss.item())

    total += loss.item()
    n += 1

  history.append(total / n)
  print(f"Epoch {epoch}: Loss {history[-1]:.4f}")

Epoch 0: Loss 2.3167
Epoch 1: Loss 2.3003
Epoch 2: Loss 2.2900


👋 ⚒ Exercise 5: Evaluate your own embedding model that has now been trained for six instead of three epochs  on the nearest neighbors of our word example list. Have the nearest neighbors further improved?

In [11]:
words = ["king", "france", "music", "water", "computer"]

for w in words:
  print(w, neighbors(w, model.center.weight.detach()))

king ['queen', 'son', 'prince', 'duke', 'alaric', 'vi', 'macedon', 'darius']
france ['portugal', 'spain', 'germany', 'italy', 'russia', 'brazil', 'bavaria', 'invaded']
music ['pop', 'dance', 'musical', 'entertainment', 'cinema', 'jazz', 'comedy', 'television']
water ['methane', 'fresh', 'dry', 'salt', 'unstable', 'yield', 'materials', 'oxide']
computer ['portable', 'computers', 'macintosh', 'hardware', 'electronic', 'computing', 'apple', 'software']


Question: Have the nearest neighbors further improved?
Answer:
Again, in short: yes. The nearest neighbors have improved after six epochs. For example "king" is now closely related to "queen". "Music" connects with words like "jazz" and "dance". "Computer" shows associations with more technology-oriented terms. However, some results especially for "water" are still less relevant.
After training, the model has generally learned more meaningful words relationships.

Load the already trained GloVe model for the next excerise.

In [12]:
glove_embeddings = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded



👋 ⚒ Exercise 6: Compare these results to the nearest neighbors of `words` in the already trained GloVe model from Tutorial 4. Which embedding model has better neighbors in your view? The tiny Skip-gram or the GloVe model?

In [14]:
for w in words:
  print(w, glove_embeddings.most_similar(w, topn=8))

king [('prince', 0.8236179351806641), ('queen', 0.7839043140411377), ('ii', 0.7746230363845825), ('emperor', 0.7736247777938843), ('son', 0.766719400882721), ('uncle', 0.7627150416374207), ('kingdom', 0.7542160749435425), ('throne', 0.7539913654327393)]
france [('french', 0.886812150478363), ('belgium', 0.8631513714790344), ('paris', 0.802532970905304), ('spain', 0.7909148335456848), ('netherlands', 0.7893800139427185), ('italy', 0.7788637280464172), ('germany', 0.7744609117507935), ('european', 0.7626755833625793)]
music [('musical', 0.8853500485420227), ('pop', 0.868208110332489), ('dance', 0.853119969367981), ('songs', 0.8525527119636536), ('recording', 0.8391881585121155), ('folk', 0.8315598964691162), ('studio', 0.8304312229156494), ('contemporary', 0.8265641927719116)]
water [('dry', 0.8274187445640564), ('natural', 0.785824179649353), ('sand', 0.7737003564834595), ('waste', 0.7724196910858154), ('drinking', 0.7561580538749695), ('clean', 0.7491700649261475), ('ocean', 0.74530559

Question: Which embedding model has better neighbors in your view? The tiny Skip-gram or the GloVe model?
Answer:
I believe the GloVe model generally produces better nearest neighbors. Both models capture meaningful word relationships but GloVe produces more consistent results. For example "france" is associated with "paris" and "french". "Computer" is associated with "software", "technology" and "internet". The small Skip-gram model also performs well although some of its neighbors seem less relevant. Generally, GloVe captures word relationships (as also mentioned in my lightning talk :)) much better.

👋 ⚒ Exercise 7: Use the analogy calculation method to evaluate potential biases in embedding models. One example of a gender bias in professions is already provided. Come up and test three more examples of bias, e.g. related to culture, gender, race, ethnicity, etc.

This shows that when training on text only, the model automatically learns bias inherent in the texts.

In [18]:
bias_analysis1 = glove_embeddings.most_similar(positive=["doctor", "woman"], negative=["man"], topn=3)
print(f"man is to doctor as woman is to {bias_analysis1}")

bias_analysis2 = glove_embeddings.most_similar(positive=["engineer", "woman"], negative=["man"])
print(f"man is to engineer as woman is to {bias_analysis2}")

bias_analysis3 = glove_embeddings.most_similar(positive=["leader", "woman"], negative=["man"])
print(f"man is to leader as woman is to {bias_analysis3}")

bias_analysis4 = glove_embeddings.most_similar(positive= ["educated", "moroccan"], negative=["french"])
print(f"french is to educated as moroccan is to {bias_analysis4}")

man is to doctor as woman is to [('nurse', 0.8404642939567566), ('child', 0.7663259506225586), ('pregnant', 0.7570130228996277)]
man is to engineer as woman is to [('technician', 0.745383083820343), ('mechanic', 0.6977542042732239), ('educator', 0.6787591576576233), ('surgeon', 0.658150315284729), ('pioneer', 0.6552696228027344), ('pioneering', 0.6509644389152527), ('chemist', 0.6468526721000671), ('physician', 0.6385414600372314), ('worker', 0.6296108365058899), ('engineering', 0.6208913326263428)]
man is to leader as woman is to [('party', 0.7881708145141602), ('opposition', 0.7746326923370361), ('lawmaker', 0.7412829995155334), ('coalition', 0.7298488616943359), ('democratic', 0.7214619517326355), ('ruling', 0.706524133682251), ('candidate', 0.7021983861923218), ('alliance', 0.7021969556808472), ('activist', 0.6970205903053284), ('president', 0.6956924200057983)]
french is to educated as moroccan is to [('schooled', 0.6715619564056396), ('naturalized', 0.6373214721679688), ("shi'a",

My observation:
The results suggest that GloVe may contain some biases. For example when comparing "man" and "doctor" with "woman" the closest results are "nurse", "child" and then even "pregnant". This pattern reflects gender stereotypes present in the training data.
When looking at "engineer" and "leader" the results include words related to jobs and politics. The bias is not as clear here. The connections seem neutral and less tied to stereotypes.
I wanted to include one race/ethnicity/nationality example: in that example, replacing "french" with "moroccan" led to words like "schooled", "naturalized", "non-muslim" and "illiterate". These results raise questions about how nationality is linked to education, religion and social status in the model.